# Gold v2 Vision Expert Final — 이어서 실행 (resume)

`lvlm_gold_v2_vision_expert_full_final_colab.ipynb`에서 **expert 계산(PatchCore / Dinomaly / AnomalyCLIP / ReConPatch-lite)이 이미 끝난 뒤** 남은 부분만 실행합니다.

하는 것:
1. Drive의 expert 결과 확인 (재계산 안 함, anomalib 설치 안 함)
2. 4 expert manifest 통합 — 이미 통합된 manifest(984행)가 있으면 그대로 재사용
3. 진행률 / 남은 시간 표시
4. 80조건 LVLM 추론 — 이미 끝난 조건·문항은 건너뛰고, 다 끝난 모델은 GPU에 로드하지도 않음
5. Oracle(GT bbox/ROI/mask) 상한선 평가
6. GitHub publish

결과 폴더와 예측 파일은 full notebook과 동일합니다(`MyDrive/lvlm_insplad_results/gold_v2_vision_expert_full/`).
끊기면 위에서부터 다시 실행하면 됩니다. GPU: L4/A100.


In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_vision_expert_full"
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"

MODELS = ["qwen3_5_4b","internvl3_5_8b","minicpm_v_4_5","llava_onevision_7b"]
EXPERTS = ["patchcore","dinomaly","anomalyclip","reconpatch_lite"]
CONDITIONS = ["score","roi","heatmap","score_roi","score_heatmap"]

CHECKPOINT_EVERY = 20  # Drive 결과 파일을 20문항마다 저장 (1이면 매 문항 전체 파일 재작성으로 느려짐)

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""  # 예: "/content/drive/MyDrive/supervised_fault_classification.zip" (비우면 자동 검색)

EXPECTED_MAIN_RUNS = len(MODELS) * len(EXPERTS) * len(CONDITIONS)  # 4 × 4 × 5 = 80


In [ ]:
import base64, csv, gc, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
QA_DIR = DRIVE_ROOT / QA_FOLDER
EXPERT_ROOT = RUN_ROOT / "experts_main"
RECON_ROOT = RUN_ROOT / "experts_reconpatch_lite"
MERGED_ROOT = RUN_ROOT / "merged_evidence"
EVAL_DIR = RUN_ROOT / "eval"
ORACLE_DIR = RUN_ROOT / "oracle"
for p in (MERGED_ROOT, EVAL_DIR, ORACLE_DIR):
    p.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""))
    return subprocess.run(cmd, env=env, check=check)

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_CONFIG_COUNT":"1",
    "GIT_CONFIG_KEY_0":"http.extraHeader",
    "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}",
    "GIT_TERMINAL_PROMPT":"0",
})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env

# LVLM 추론만 하므로 anomalib 등 expert 패키지는 설치하지 않습니다.
run([sys.executable,"-m","pip","install","-q",
     "transformers==5.14.0","accelerate==1.14.0","sentencepiece==0.2.2",
     "pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))

import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")


In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음). 경로를 알면 설정 셀의 *_ZIP_PATH에 넣으세요.", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

def read_jsonl(path):
    return [json.loads(x) for x in Path(path).read_text(encoding="utf-8").splitlines() if x.strip()]

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
print("SUP:", SUP_ZIP)
if not sorted(QA_DIR.glob("master_qa_*.jsonl")):
    raise FileNotFoundError(f"master_qa_*.jsonl under {QA_DIR}")

# Expert 계산 결과 확인 — 이 notebook은 expert를 다시 계산하지 않습니다.
def expert_queries(path, expert):
    if not path.is_file():
        return 0
    df = pd.read_csv(path)
    return df.loc[df["expert"] == expert, "query_id"].astype(str).nunique()

expert_counts = {e: expert_queries(EXPERT_ROOT / "roi_manifest.csv", e) for e in ("patchcore","dinomaly","anomalyclip")}
expert_counts["reconpatch_lite"] = expert_queries(RECON_ROOT / "roi_manifest_reconpatch_lite.csv", "reconpatch_lite")
print("expert 완료 query 수:", expert_counts)
merged_exists = (MERGED_ROOT / "roi_manifest_all.csv").is_file()
missing = [e for e in EXPERTS if expert_counts.get(e, 0) < 246]
if missing and not merged_exists:
    raise RuntimeError(
        f"expert 결과가 덜 끝났습니다: {missing}. "
        "lvlm_gold_v2_vision_expert_full_final_colab.ipynb로 expert 단계부터 실행하세요."
    )


In [ ]:
# 4 expert manifest 통합 + 순수 heatmap overlay 생성
# 기존 overlay에는 yellow ROI box가 함께 있어 heatmap 조건을 confound하므로,
# 최종 실험에서는 box 없는 pure heatmap overlay를 별도로 생성해 overlay_path로 사용합니다.
# 재실행 시: 통합 manifest(984행)가 이미 있으면 그대로 재사용하고,
# 없을 때도 이미 만든 heatmap PNG는 다시 읽거나 합성하지 않습니다.
import numpy as np
from PIL import Image
from insplad_benchmark.vision_expert import normalize_anomaly_map

FORCE_REMERGE = False  # expert를 다시 계산했을 때만 True
MERGED_MANIFEST = MERGED_ROOT / "roi_manifest_all.csv"
PURE_PROTOCOL = "pure_heatmap_overlay_without_bbox"
EXPECTED_MERGED = 246 * len(EXPERTS)

def read_csv_rows(path):
    with Path(path).open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def abs_artifact(root, value):
    p = Path(str(value))
    return p if p.is_absolute() else (Path(root) / p).resolve()

def merged_manifest_ready():
    if not MERGED_MANIFEST.is_file():
        return False, "통합 manifest 없음"
    rows = [r for r in read_csv_rows(MERGED_MANIFEST) if r.get("expert") in EXPERTS]
    keys = {(str(r["query_id"]), str(r["expert"])) for r in rows}
    if len(keys) != EXPECTED_MERGED:
        return False, f"통합 manifest coverage {len(keys)} != {EXPECTED_MERGED}"
    if any(r.get("heatmap_evidence_protocol") != PURE_PROTOCOL for r in rows):
        return False, "pure heatmap protocol이 아닌 행 있음"
    return True, f"{len(keys)}행"

ok, reason = merged_manifest_ready()
if ok and not FORCE_REMERGE:
    print("재사용:", MERGED_MANIFEST, reason)
else:
    print("통합 실행:", reason)
    main_manifest = EXPERT_ROOT / "roi_manifest.csv"
    recon_manifest = RECON_ROOT / "roi_manifest_reconpatch_lite.csv"
    all_rows = []
    for manifest, root in ((main_manifest, EXPERT_ROOT), (recon_manifest, RECON_ROOT)):
        for row in read_csv_rows(manifest):
            if row.get("expert") not in EXPERTS:
                continue
            row = dict(row)
            for field in ("anomaly_map_path","roi_path","overlay_path","heatmap_path"):
                if row.get(field):
                    row[field] = str(abs_artifact(root, row[field]))
            all_rows.append(row)

    index = {(str(r["query_id"]), str(r["expert"])): r for r in all_rows}
    if len(index) != EXPECTED_MERGED:
        raise RuntimeError(f"{len(EXPERTS)}-expert manifest coverage: {len(index)} != {EXPECTED_MERGED}")

    pure_root = MERGED_ROOT / "pure_heatmap"
    created = reused = 0
    for i, row in enumerate(all_rows, 1):
        out = pure_root / str(row["expert"]) / f"{row['query_id']}__pure_heatmap.png"
        row["overlay_with_roi_box_path"] = row.get("overlay_path","")
        row["overlay_path"] = str(out.resolve())
        row["heatmap_evidence_protocol"] = PURE_PROTOCOL
        if out.is_file() and out.stat().st_size > 0:
            reused += 1
        else:
            amap = np.load(Path(row["anomaly_map_path"]), allow_pickle=False)
            norm = normalize_anomaly_map(amap)
            heat = np.zeros((*norm.shape, 3), dtype=np.uint8)
            heat[..., 0] = np.rint(255 * norm).astype(np.uint8)
            heat[..., 1] = np.rint(160 * np.square(norm)).astype(np.uint8)
            heat_img = Image.fromarray(heat)
            with Image.open(Path(row["source_image_path"])) as h:
                target = h.convert("RGB")
            heat_img = heat_img.resize(target.size, Image.Resampling.BILINEAR)
            overlay = Image.blend(target, heat_img, alpha=0.42)
            out.parent.mkdir(parents=True, exist_ok=True)
            tmp = out.with_name(out.stem + ".tmp.png")
            overlay.save(tmp, format="PNG")
            tmp.replace(out)  # 중간에 끊겨도 깨진 PNG가 남지 않음
            created += 1
        if i % 100 == 0 or i == len(all_rows):
            print(f"heatmap {i}/{len(all_rows)} (새로 생성 {created}, 재사용 {reused})", flush=True)

    fields = []
    for row in all_rows:
        for k in row:
            if k not in fields:
                fields.append(k)
    tmp_manifest = MERGED_MANIFEST.with_name(MERGED_MANIFEST.name + ".tmp")
    with tmp_manifest.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(all_rows)
    tmp_manifest.replace(MERGED_MANIFEST)
    print("merged:", len(all_rows), MERGED_MANIFEST)


In [ ]:
# 80조건 진행률 / 남은 시간 (Drive prediction 기준, 추론 없음)
import statistics

def show_progress():
    n = 534
    total_left = 0.0
    done_runs = 0
    for m in MODELS:
        lat, left, full = [], 0, 0
        for e in EXPERTS:
            for c in CONDITIONS:
                p = EVAL_DIR / "predictions" / m / f"{e}__{c}.jsonl"
                rows = [json.loads(x) for x in p.read_text(encoding="utf-8").splitlines() if x.strip()] if p.exists() else []
                ok = [r for r in rows if not r.get("error") and str(r.get("output","")).strip()]
                lat += [float(r["latency_s"]) for r in ok if r.get("latency_s")]
                left += max(0, n - len(ok))
                full += len(ok) >= n
        sec = statistics.mean(lat) if lat else 3.0
        total_left += left * sec
        done_runs += full
        print(f"{m:20s} 완료 {full:2d}/{len(EXPERTS)*len(CONDITIONS)}  남은 문항 {left:5d}  문항당 {sec:.2f}s  ≈ {left*sec/3600:.1f}시간")
    print(f"\n전체 {done_runs}/{EXPECTED_MAIN_RUNS} 조건 완료, 남은 LVLM 추론 ≈ {int(total_left//3600)}시간 {int(total_left%3600//60)}분 (+Oracle 약 30분)")

show_progress()


In [ ]:
# 최종 80조건: 4 experts × 5 evidence × 4 LVLM
# prediction JSONL은 조건별 Drive checkpoint라 같은 셀 재실행 시 완료된 응답 재사용.
run([
    sys.executable, REPO_DIR / "scripts/run_gold_v2_vision_expert_full.py",
    "--qa-dir", QA_DIR,
    "--sup-zip", SUP_ZIP,
    "--roi-manifest", MERGED_MANIFEST,
    "--artifact-root", MERGED_ROOT,
    "--out-dir", EVAL_DIR,
    "--models", *MODELS,
    "--experts", *EXPERTS,
    "--conditions", *CONDITIONS,
    "--checkpoint-every", str(CHECKPOINT_EVERY),
])

leaderboard = pd.read_csv(EVAL_DIR / "leaderboard.csv")
coverage = leaderboard[
    leaderboard.model_key.isin(MODELS)
    & leaderboard.expert.isin(EXPERTS)
    & leaderboard.condition.isin(CONDITIONS)
].drop_duplicates(["model_key","expert","condition"])
if len(coverage) != EXPECTED_MAIN_RUNS:
    raise RuntimeError(f"final coverage {len(coverage)} != {EXPECTED_MAIN_RUNS}")
display(pd.read_csv(EVAL_DIR / "screening_summary.csv").head(20))

show_progress()


## Oracle localization evidence

아래 단계는 deployable Vision Expert 80-condition ranking과 별개입니다.

- GT bbox
- GT ROI crop
- GT polygon/mask overlay

를 사용해 “완벽한 위치 정보가 주어졌을 때 LVLM이 얼마나 좋아지는가?”를 봅니다.

정상 이미지에서 empty GT를 주면 anomaly 정답이 노출되므로,
non-empty defect localization GT가 있는 이미지의 `Q2 defect classification + Q3 localization`만 사용합니다.


In [ ]:
# GT bbox / ROI / mask artifact 생성
from PIL import Image, ImageDraw
from zipfile import ZipFile

loc_files = sorted(QA_DIR.glob("localization_qa_*.jsonl"))
if not loc_files:
    raise FileNotFoundError("localization_qa_*.jsonl")
loc_rows_all = read_jsonl(loc_files[-1])

# 같은 core의 localization_gt metadata를 하나로 모음
loc_meta = {}
for row in loc_rows_all:
    gt = row.get("localization_gt") or {}
    if gt and not bool(gt.get("empty_mask", False)) and gt.get("normalized_bboxes_xywh"):
        loc_meta[str(row["core_id"])] = row

print("non-empty localization GT images:", len(loc_meta))

def extract_member(zip_path, member, root):
    out = root / member
    if out.is_file():
        return out
    out.parent.mkdir(parents=True, exist_ok=True)
    with ZipFile(zip_path) as z:
        try:
            data = z.read(member)
        except KeyError:
            norm = member.replace("\\","/").lstrip("/")
            actual = next(n for n in z.namelist() if n.replace("\\","/").lstrip("/") == norm)
            data = z.read(actual)
    out.write_bytes(data)
    return out

oracle_rows = []
for core_id, row in loc_meta.items():
    target_path = extract_member(SUP_ZIP, str(row["image"]), ORACLE_DIR / "_images")
    done_dir = ORACLE_DIR / "visual" / core_id
    done_paths = [done_dir / "gt_bbox.png", done_dir / "gt_roi.png", done_dir / "gt_mask_overlay.png"]
    if all(p.is_file() and p.stat().st_size > 0 for p in done_paths):
        # 재실행 시 이미 만든 oracle 이미지는 재사용
        oracle_rows.append({
            "core_id": core_id,
            "target_path": str(target_path.resolve()),
            "gt_bbox_path": str(done_paths[0].resolve()),
            "gt_roi_path": str(done_paths[1].resolve()),
            "gt_mask_path": str(done_paths[2].resolve()),
            "coarse_region": row["localization_gt"].get("coarse_region",""),
        })
        continue
    with Image.open(target_path) as h:
        img = h.convert("RGB")
    W, H = img.size
    gt = row["localization_gt"]
    boxes = gt["normalized_bboxes_xywh"]
    px_boxes = []
    for x,y,w,h in boxes:
        left = int(round(float(x)*W)); top = int(round(float(y)*H))
        right = int(round((float(x)+float(w))*W)); bottom = int(round((float(y)+float(h))*H))
        left=max(0,min(left,W-1)); top=max(0,min(top,H-1))
        right=max(left+1,min(right,W)); bottom=max(top+1,min(bottom,H))
        px_boxes.append((left,top,right,bottom))
    union = (
        min(b[0] for b in px_boxes), min(b[1] for b in px_boxes),
        max(b[2] for b in px_boxes), max(b[3] for b in px_boxes)
    )

    out_dir = ORACLE_DIR / "visual" / core_id
    out_dir.mkdir(parents=True, exist_ok=True)

    bbox_img = img.copy()
    d = ImageDraw.Draw(bbox_img)
    lw = max(3, round(min(W,H)/180))
    for b in px_boxes:
        d.rectangle((b[0],b[1],b[2]-1,b[3]-1), outline=(255,0,0), width=lw)
    bbox_path = out_dir / "gt_bbox.png"
    bbox_img.save(bbox_path)

    roi_path = out_dir / "gt_roi.png"
    img.crop(union).save(roi_path)

    mask = Image.new("L", (W,H), 0)
    md = ImageDraw.Draw(mask)
    polygons = gt.get("normalized_polygons_xy") or []
    if polygons:
        for poly in polygons:
            pts = [(int(round(float(x)*W)), int(round(float(y)*H))) for x,y in poly]
            if len(pts) >= 3:
                md.polygon(pts, fill=180)
    else:
        for b in px_boxes:
            md.rectangle((b[0],b[1],b[2]-1,b[3]-1), fill=180)
    red = Image.new("RGB",(W,H),(255,0,0))
    mask_overlay = Image.composite(red, img, mask)
    mask_overlay = Image.blend(img, mask_overlay, alpha=0.45)
    mask_path = out_dir / "gt_mask_overlay.png"
    mask_overlay.save(mask_path)

    oracle_rows.append({
        "core_id": core_id,
        "target_path": str(target_path.resolve()),
        "gt_bbox_path": str(bbox_path.resolve()),
        "gt_roi_path": str(roi_path.resolve()),
        "gt_mask_path": str(mask_path.resolve()),
        "coarse_region": gt.get("coarse_region",""),
    })

ORACLE_MANIFEST = ORACLE_DIR / "oracle_manifest.csv"
pd.DataFrame(oracle_rows).to_csv(ORACLE_MANIFEST, index=False)
display(pd.DataFrame(oracle_rows).head())


In [ ]:
# Oracle Q2/Q3 × 3 evidence × 4 LVLM
import time, traceback
sys.path.insert(0, str(REPO_DIR / "scripts"))
import run_gold_baselines as base
from insplad_benchmark.hf_adapters import load_hf_adapter, infer_hf

oracle_index = {r["core_id"]: r for r in oracle_rows}
oracle_qa = [
    r for r in loc_rows_all
    if str(r.get("core_id")) in oracle_index
    and r.get("question_type") in {"defect_classification","defect_localization"}
]
oracle_conditions = ["gt_bbox","gt_roi","gt_mask"]

def oracle_prompt(row, condition):
    options = "\n".join(f"{k}. {v}" for k,v in row["options"].items())
    desc = {
        "gt_bbox":"Image 2 shows the ground-truth defect bounding box.",
        "gt_roi":"Image 2 is a crop of the ground-truth defect region.",
        "gt_mask":"Image 2 shows the ground-truth defect mask overlay.",
    }[condition]
    return (
        "You are an industrial visual inspector. Image 1 is the complete TARGET image. "
        + desc +
        " This is ORACLE upper-bound evidence, not a deployable model output. "
        "Answer only the multiple-choice question.\n\n"
        f"Question: {row['question']}\n{options}\n\n"
        "Return exactly one option letter and nothing else."
    )

def oracle_images(row, condition):
    info = oracle_index[str(row["core_id"])]
    key = {"gt_bbox":"gt_bbox_path","gt_roi":"gt_roi_path","gt_mask":"gt_mask_path"}[condition]
    return [Path(info["target_path"]), Path(info[key])]

def pred_path_for(model, condition):
    return ORACLE_DIR / "predictions" / model / f"{condition}.jsonl"

oracle_metrics = []
for model_key in MODELS:
    spec = base.MODEL_REGISTRY[model_key]
    # 첫 미완료 문항에서만 모델 로드 — 이미 끝난 모델은 로드하지 않음
    holder = base.LazyHFAdapter(spec, loader=load_hf_adapter)
    try:
        for condition in oracle_conditions:
            pred_path = pred_path_for(model_key, condition)
            existing = {str(r["id"]):r for r in read_jsonl(pred_path)} if pred_path.is_file() else {}
            preds = []
            called = 0
            for i,row in enumerate(oracle_qa,1):
                old = existing.get(str(row["id"]))
                if old and not old.get("error") and str(old.get("output","")).strip():
                    preds.append(old); continue
                images = oracle_images(row, condition)
                prompt_text = oracle_prompt(row, condition)
                adapter = holder.get()
                started=time.time(); output=""; error=""
                try:
                    output = str(infer_hf(adapter, images, prompt_text))
                except Exception as exc:
                    error=f"{type(exc).__name__}: {exc}"
                choice = base.parse_choice(output, {str(k).upper() for k in row["options"]})
                preds.append({
                    "id":row["id"],"core_id":row["core_id"],"question_type":row["question_type"],
                    "model_key":model_key,"condition":condition,"output":output,
                    "predicted_choice":choice,"latency_s":round(time.time()-started,3),"error":error,
                })
                called += 1
                if called % CHECKPOINT_EVERY == 0:
                    base.write_jsonl(pred_path, preds)
            base.write_jsonl(pred_path, preds)

            by_id={str(r["id"]):r for r in preds}
            q2=[r for r in oracle_qa if r["question_type"]=="defect_classification"]
            q3=[r for r in oracle_qa if r["question_type"]=="defect_localization"]
            q2_gold=[r["defect_gt"] for r in q2]
            q2_pred=[base.predicted_semantic(r, by_id[str(r["id"])].get("predicted_choice")) for r in q2]
            metric={
                "model_key":model_key,"condition":condition,
                "images":len(oracle_index),"questions":len(oracle_qa),
                "valid_output_rate":sum(by_id[str(r["id"])].get("predicted_choice") is not None for r in oracle_qa)/len(oracle_qa),
                "overall_accuracy":sum(by_id[str(r["id"])].get("predicted_choice")==r["answer"] for r in oracle_qa)/len(oracle_qa),
                "defect_accuracy":sum(by_id[str(r["id"])].get("predicted_choice")==r["answer"] for r in q2)/len(q2) if q2 else None,
                "defect_macro_f1":base.macro_f1(q2_gold,q2_pred) if q2 else None,
                "localization_accuracy":sum(by_id[str(r["id"])].get("predicted_choice")==r["answer"] for r in q3)/len(q3) if q3 else None,
            }
            oracle_metrics.append(metric)
            pd.DataFrame(oracle_metrics).to_csv(ORACLE_DIR/"oracle_leaderboard.csv",index=False)
            print(metric)
    finally:
        adapter=None
        holder.release()

display(pd.DataFrame(oracle_metrics))


In [ ]:
# 핵심 결과 GitHub publish
publish_files = [
    "leaderboard.csv","metrics_by_asset.csv","run_status.json",
    "screening_summary.csv","run_metadata.json",
]
run([
    sys.executable, REPO_DIR / "scripts/publish_results_to_github.py",
    "--out-dir", EVAL_DIR,
    "--repo", REPO,
    "--branch", REPO_REF,
    "--prefix", "gold_v2_vision_expert_full",
    "--files", *[f for f in publish_files if (EVAL_DIR/f).is_file()],
], env=os.environ.copy())

# Oracle summary도 별도 prefix
oracle_publish = ["oracle_leaderboard.csv","oracle_manifest.csv"]
run([
    sys.executable, REPO_DIR / "scripts/publish_results_to_github.py",
    "--out-dir", ORACLE_DIR,
    "--repo", REPO,
    "--branch", REPO_REF,
    "--prefix", "gold_v2_vision_expert_oracle",
    "--files", *[f for f in oracle_publish if (ORACLE_DIR/f).is_file()],
], env=os.environ.copy())

print("메인 완료:", EVAL_DIR)
print("Oracle 완료:", ORACLE_DIR)


## 완료 후

- GitHub `results/latest/gold_v2_vision_expert_full/`과 `results/latest/gold_v2_vision_expert_oracle/`에 결과가 올라갑니다.
- 해석 시:
  - 50장 smoke는 preliminary
  - 246장 80조건이 최종 Vision Expert ablation
  - GT bbox/ROI/mask는 4 HF LVLM의 oracle upper-bound로 별도 표기
- 다음 실행: `notebooks/lvlm_gold_v2_reference_expert_colab.ipynb` (Normal Reference + 상위 Vision Expert 결합)
